# Notebook 02: First Baseline Model (Load from Google Drive)
Dataset: EEG_Scaled_data.csv (2.65 GB) from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --quiet
!pip install pandas numpy scikit-learn matplotlib seaborn --quiet

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print("✅ Drive mounted + Libraries ready")
print("GPU available:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted + Libraries ready
GPU available: False


In [ ]:
file_path = "/content/drive/MyDrive/EEG_Scaled_data.csv"

df = pd.read_csv(file_path)
print("✅ File loaded successfully from Drive")
print("Shape:", df.shape)
print("Columns preview:", df.columns.tolist()[:15], "...")
print("Target column:", 'target' in df.columns)
print("Target distribution:", df['target'].value_counts().to_dict())

✅ File loaded successfully from Drive
Shape: (11233, 36865)
Columns preview: ['Channel_1', 'Channel_2', 'Channel_3', 'Channel_4', 'Channel_5', 'Channel_6', 'Channel_7', 'Channel_8', 'Channel_9', 'Channel_10', 'Channel_11', 'Channel_12', 'Channel_13', 'Channel_14', 'Channel_15'] ...
Target column: True
Target distribution: {0: 9799, 1: 1434}


In [ ]:
# Drop target column for features
X = df.drop(columns=['target']).values.astype(np.float32)
y = df['target'].values.astype(np.int64)

print(f"Features shape: {X.shape} | Labels: {np.bincount(y)}")

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx].unsqueeze(0), self.y[idx]   # (1, features) for Conv1d

dataset = EEGDataset(X, y)
print(f"Dataset ready with {len(dataset)} samples")

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, stride=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, stride=2)
        self.pool = nn.MaxPool1d(2)
        self.fc1 = nn.Linear(64 * (input_size // 8), 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):
        x = x.unsqueeze(1)   # (batch, 1, features)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(input_size=X.shape[1]).to(device)
print("✅ Model created")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

all_preds, all_labels = [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"\n--- Fold {fold+1}/5 ---")
    train_loader = DataLoader(torch.utils.data.Subset(dataset, train_idx), batch_size=64, shuffle=True)
    val_loader = DataLoader(torch.utils.data.Subset(dataset, val_idx), batch_size=64, shuffle=False)

    for epoch in range(6):   # 6 epochs for better convergence
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch_y.numpy())

acc = accuracy_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_preds)

print("\n🎉 FIRST BASELINE RESULTS")
print(f"Accuracy : {acc:.4f}")
print(f"AUC      : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Non-Seizure (0)', 'Seizure (1)']))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - First Baseline Model')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

torch.save(model.state_dict(), "/content/baseline_model.pth")
print("✅ Model saved")

from google.colab import files
files.download("/content/baseline_model.pth")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - First Baseline Model')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Save model to Colab (temporary)
torch.save(model.state_dict(), "/content/baseline_model.pth")
print("✅ Model saved locally as baseline_model.pth")

# Save model permanently to Google Drive
drive_model_path = "/content/drive/MyDrive/seizure_prediction_data/baseline_model.pth"

torch.save(model.state_dict(), drive_model_path)
print(f"✅ Model also saved permanently to Google Drive:")
print(f"   {drive_model_path}")

# Optional: Download locally too
from google.colab import files
files.download("/content/baseline_model.pth")